# Pipeline walkthrough
Try each part of the pipeline on one sound: the filter, the filter ID guess, undoing the filter, and the event model. Change `pipeline-walkthrough-settings.json`, then rerun the cells.

- `sound`: what to test on. `kind` is `wav` (a piece of `wav` from `start_s`), `tone` or `thump` (at `freq_hz`), or `noise`.
- `filter`: the filter to apply. `type` is `highpass`, `lowpass`, or `bandpass`, and a band-pass gets two cutoffs and an even `order`.
- `noise`: `snr_db` (`null` for none) and `stage` (`before_filter` or `after_filter`). `bits` is the rounding.
- `filter_id_model`: a trained filter ID model.
- `undo`: `method` is `python`, or `c` on Linux with the C program built at `c_command`.
- `out_dir`: where the WAVs go.
- `event_model`: its settings file, the voiceover folder, the run folder, the seed and clip counts, which test clip to plot, and the WAV to find events in.

In [ ]:
from pipeline import walkthrough as wt

%matplotlib inline

cfg = wt.read_settings("pipeline-walkthrough-settings.json")

## 1. The test sound
Scaled to the level the filter ID model trained at.

In [ ]:
clean = wt.make_sound(cfg)

## 2. Damage it
The filter, then the noise, then rounding to the bit depth.

In [ ]:
damaged, steps = wt.damage(clean, cfg)

## 3. Guess the filter
The filter ID model only sees the damaged sound.

In [ ]:
guess = wt.guess_filter(damaged, cfg)
filter_figure = wt.plot_filters(cfg, guess)

## 4. Undo the filter
Undone with the real settings and with the guess. Every version gets saved as a 32-bit float WAV in `out_dir` so you can listen to it in a DAW.

In [ ]:
versions = wt.undo_filter(clean, damaged, guess, cfg)
spectra_figure = wt.plot_spectra(versions, cfg)
wav_paths = wt.save_wavs(versions, cfg)

## 5. Train or load the event model
The first run trains on the team's voiceovers, which takes a while. After that it loads the saved model. A different seed or clip count needs a new `run_dir`.

In [ ]:
em_cfg = cfg["event_model"]
em_settings = wt.event_model_settings(cfg)
model, em_settings = wt.train_or_load(em_settings, em_cfg["run_dir"])

## 6. One test clip
A clip from the held-out voiceover. Green is a hit, orange is a missed event, and red is a false alarm.

In [ ]:
audio, record, detected = wt.make_test_clip(model, em_settings, em_cfg["run_dir"], em_cfg["test_clip"])
clip_figure = wt.plot_test_clip(audio, record, detected, em_settings)

## 7. Four-version comparison
The same held-out cases, scored clean, filtered, with the known filter undone, and with the AI's guess undone. Without a filter ID model it skips the last one. A score shows as `n/a` when there's nothing to divide by.

In [ ]:
comparison = wt.compare_versions(
    em_cfg["run_dir"], cfg["filter_id_model"], cfg["undo"]["method"], cfg["undo"]["c_command"]
)
f1_figure = wt.show_comparison(comparison)

## 8. Events in a WAV
Any mono 48 kHz WAV works.

In [ ]:
events = wt.detect_file(em_cfg["run_dir"], em_cfg["detect_wav"])
wt.print_events(events)

## 9. Run the tests
The whole test suite. 0 means everything passed.

In [ ]:
wt.run_tests()